In [6]:
import pandas as pd
import numpy as np
import warnings
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, f1_score, recall_score,
                              precision_score, classification_report)
import xgboost as xgb
import lightgbm as lgb

In [ ]:
INPUT_PATH  = r"Membership_v3.xlsx"
OUTPUT_PATH = r"Membership_v3(all_model_results).xlsx"

In [8]:
df = pd.read_excel(INPUT_PATH)

df["plan_tier_enc"] = df["plan_tier"].map(
    {"기타": 0, "basic": 1, "standard": 2, "premium": 3}
)
df["currency_enc"] = (df["currency_type"] == "USD").astype(int)

device_dummies = pd.get_dummies(df["payment_device"], prefix="dev", drop_first=False)
df = pd.concat([df, device_dummies], axis=1)

In [9]:
BASE_FEATURES = [
    "concurrent_streams", "is_user_verified", "gender", "age",
    "duration_days", "is_promotional_price", "is_night_signup",
    "reg_weekday", "amt_7900", "amt_10900", "amt_13900",
    "age_group", "has_watch_in_period", "first_watch_offset",
    "watch_density", "last_watch_offset",
    "plan_tier_enc", "currency_enc"
] + list(device_dummies.columns)

CASE_A = BASE_FEATURES + ["is_churn_prevented"]  # is_churn_prevented 포함
CASE_B = BASE_FEATURES                            # is_churn_prevented 제외

TARGET = "repurchase"
y      = df[TARGET]
spw    = float(y.value_counts()[0] / y.value_counts()[1])  # 클래스 불균형 보정값

In [10]:
RANDOM_STATE = 42
TEST_SIZE    = 0.2
SKF          = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [11]:
def get_default_models(spw):
    return {
        "LogisticRegression": LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "DecisionTree": DecisionTreeClassifier(
            class_weight="balanced", random_state=RANDOM_STATE
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=100, class_weight="balanced",
            n_jobs=-1, random_state=RANDOM_STATE
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=100, random_state=RANDOM_STATE
        ),
        "XGBoost": xgb.XGBClassifier(
            scale_pos_weight=spw, eval_metric="logloss",
            random_state=RANDOM_STATE, verbosity=0
        ),
        "LightGBM": lgb.LGBMClassifier(
            class_weight="balanced", random_state=RANDOM_STATE, verbose=-1
        ),
    }

In [12]:
def evaluate(model, X_te, y_te):
    prob = model.predict_proba(X_te)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        "AUC":       roc_auc_score(y_te, prob),
        "F1":        f1_score(y_te, pred),
        "Recall":    recall_score(y_te, pred),
        "Precision": precision_score(y_te, pred),
        "Accuracy":  (pred == y_te).mean(),
    }

In [13]:
# ── Step 1. 기본 파라미터 전체 모델 비교 ──────────────────
print("=" * 72)
print("  Step 1. 기본 파라미터 비교 (test set)")
print("=" * 72)
print(f"  {'모델':<22} {'Case':<7} {'AUC':>8} {'F1':>7} {'Recall':>8} {'Prec':>8}")
print("=" * 72)

step1_rows = []
for case_name, features in [("A", CASE_A), ("B", CASE_B)]:
    X      = df[features].fillna(0)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    scaler = StandardScaler()
    X_tr_s = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns)
    X_te_s = pd.DataFrame(scaler.transform(X_te),     columns=X_te.columns)

    for mname, model in get_default_models(spw).items():
        Xi_tr = X_tr_s if mname == "LogisticRegression" else X_tr
        Xi_te = X_te_s if mname == "LogisticRegression" else X_te
        model.fit(Xi_tr, y_tr)
        m = evaluate(model, Xi_te, y_te)
        row = {"Case": case_name, "Model": mname, "Tuned": False, **m}
        step1_rows.append(row)
        print(f"  {mname:<22} Case{case_name:<6} "
              f"{m['AUC']:>8.4f} {m['F1']:>7.4f} "
              f"{m['Recall']:>8.4f} {m['Precision']:>8.4f}")

print("=" * 72)

  Step 1. 기본 파라미터 비교 (test set)
  모델                     Case         AUC      F1   Recall     Prec
  LogisticRegression     CaseA        0.6318  0.6735   0.6186   0.7391
  DecisionTree           CaseA        0.5579  0.6924   0.6821   0.7030
  RandomForest           CaseA        0.6099  0.7651   0.8475   0.6973
  GradientBoosting       CaseA        0.6597  0.8026   0.9420   0.6992
  XGBoost                CaseA        0.6289  0.6963   0.6602   0.7366


  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_c

  LightGBM               CaseA        0.6457  0.7023   0.6619   0.7480
  LogisticRegression     CaseB        0.6297  0.6740   0.6186   0.7402
  DecisionTree           CaseB        0.5543  0.6935   0.6863   0.7010
  RandomForest           CaseB        0.6025  0.7563   0.8295   0.6949
  GradientBoosting       CaseB        0.6512  0.8043   0.9530   0.6958
  XGBoost                CaseB        0.6216  0.6964   0.6627   0.7336
  LightGBM               CaseB        0.6399  0.6966   0.6548   0.7442


In [14]:
# ── Step 2. Optuna 하이퍼파라미터 최적화 ──────────────────
# 대상 모델: XGBoost, LightGBM, GradientBoosting
# n_trials 조정 가능 (높을수록 정밀, 느림)
N_TRIALS = 50

print(f"\n{'='*72}")
print(f"  Step 2. Optuna 최적화 ({N_TRIALS} trials/모델)")
print(f"{'='*72}")

step2_rows = []

for case_name, features in [("A", CASE_A), ("B", CASE_B)]:
    X = df[features].fillna(0)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    print(f"\n  [Case {case_name}]")

    # ── XGBoost ──────────────────────────────────────────
    def obj_xgb(trial):
        p = dict(
            n_estimators     = trial.suggest_int("n_estimators", 100, 500),
            max_depth        = trial.suggest_int("max_depth", 2, 8),
            learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            subsample        = trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0),
            min_child_weight = trial.suggest_int("min_child_weight", 1, 10),
            scale_pos_weight = spw,
            eval_metric="logloss", random_state=RANDOM_STATE, verbosity=0
        )
        return cross_val_score(
            xgb.XGBClassifier(**p), X_tr, y_tr,
            cv=SKF, scoring="roc_auc", n_jobs=-1
        ).mean()

    study_xgb = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study_xgb.optimize(obj_xgb, n_trials=N_TRIALS, show_progress_bar=False)
    bp = study_xgb.best_trial.params
    best_xgb = xgb.XGBClassifier(
        n_estimators=bp["n_estimators"], max_depth=bp["max_depth"],
        learning_rate=bp["learning_rate"], subsample=bp["subsample"],
        colsample_bytree=bp["colsample_bytree"], min_child_weight=bp["min_child_weight"],
        scale_pos_weight=spw, eval_metric="logloss",
        random_state=RANDOM_STATE, verbosity=0
    )
    best_xgb.fit(X_tr, y_tr)
    m = evaluate(best_xgb, X_te, y_te)
    row = {"Case": case_name, "Model": "XGBoost(Optuna)", "Tuned": True,
           "CV_AUC": study_xgb.best_value, "Best_Params": str(bp), **m}
    step2_rows.append(row)
    print(f"    XGBoost      CV_AUC={study_xgb.best_value:.4f}  "
          f"Test_AUC={m['AUC']:.4f}  Recall={m['Recall']:.4f}")

    # ── LightGBM ─────────────────────────────────────────
    def obj_lgb(trial):
        p = dict(
            n_estimators     = trial.suggest_int("n_estimators", 100, 500),
            max_depth        = trial.suggest_int("max_depth", 2, 8),
            learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            num_leaves       = trial.suggest_int("num_leaves", 20, 100),
            subsample        = trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0),
            min_child_samples= trial.suggest_int("min_child_samples", 10, 50),
            class_weight="balanced", random_state=RANDOM_STATE, verbose=-1
        )
        return cross_val_score(
            lgb.LGBMClassifier(**p), X_tr, y_tr,
            cv=SKF, scoring="roc_auc", n_jobs=-1
        ).mean()

    study_lgb = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study_lgb.optimize(obj_lgb, n_trials=N_TRIALS, show_progress_bar=False)
    blp = study_lgb.best_trial.params
    best_lgb = lgb.LGBMClassifier(
        n_estimators=blp["n_estimators"], max_depth=blp["max_depth"],
        learning_rate=blp["learning_rate"], num_leaves=blp["num_leaves"],
        subsample=blp["subsample"], colsample_bytree=blp["colsample_bytree"],
        min_child_samples=blp["min_child_samples"],
        class_weight="balanced", random_state=RANDOM_STATE, verbose=-1
    )
    best_lgb.fit(X_tr, y_tr)
    m = evaluate(best_lgb, X_te, y_te)
    row = {"Case": case_name, "Model": "LightGBM(Optuna)", "Tuned": True,
           "CV_AUC": study_lgb.best_value, "Best_Params": str(blp), **m}
    step2_rows.append(row)
    print(f"    LightGBM     CV_AUC={study_lgb.best_value:.4f}  "
          f"Test_AUC={m['AUC']:.4f}  Recall={m['Recall']:.4f}")

    # ── GradientBoosting ──────────────────────────────────
    def obj_gb(trial):
        p = dict(
            n_estimators    = trial.suggest_int("n_estimators", 100, 500),
            max_depth       = trial.suggest_int("max_depth", 2, 6),
            learning_rate   = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            subsample       = trial.suggest_float("subsample", 0.6, 1.0),
            min_samples_leaf= trial.suggest_int("min_samples_leaf", 10, 50),
            random_state    = RANDOM_STATE
        )
        return cross_val_score(
            GradientBoostingClassifier(**p), X_tr, y_tr,
            cv=SKF, scoring="roc_auc", n_jobs=-1
        ).mean()

    study_gb = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study_gb.optimize(obj_gb, n_trials=N_TRIALS, show_progress_bar=False)
    bgp = study_gb.best_trial.params
    best_gb = GradientBoostingClassifier(**bgp)
    best_gb.fit(X_tr, y_tr)
    m = evaluate(best_gb, X_te, y_te)
    row = {"Case": case_name, "Model": "GradientBoosting(Optuna)", "Tuned": True,
           "CV_AUC": study_gb.best_value, "Best_Params": str(bgp), **m}
    step2_rows.append(row)
    print(f"    GradBoosting CV_AUC={study_gb.best_value:.4f}  "
          f"Test_AUC={m['AUC']:.4f}  Recall={m['Recall']:.4f}")


  Step 2. Optuna 최적화 (50 trials/모델)

  [Case A]
    XGBoost      CV_AUC=0.6628  Test_AUC=0.6577  Recall=0.6766
    LightGBM     CV_AUC=0.6619  Test_AUC=0.6572  Recall=0.6737
    GradBoosting CV_AUC=0.6624  Test_AUC=0.6559  Recall=0.9811

  [Case B]
    XGBoost      CV_AUC=0.6557  Test_AUC=0.6480  Recall=0.6644
    LightGBM     CV_AUC=0.6563  Test_AUC=0.6505  Recall=0.6678
    GradBoosting CV_AUC=0.6555  Test_AUC=0.6506  Recall=0.9567


In [15]:
all_rows = step1_rows + step2_rows
res = pd.DataFrame(all_rows)

print(f"\n{'='*72}")
print("  최종 결과 요약 (AUC 내림차순)")
print(f"{'='*72}")
print(f"  {'모델':<28} {'Case':<7} {'AUC':>8} {'F1':>7} {'Recall':>8} {'Prec':>8}")
print(f"{'='*72}")
for case in ["A", "B"]:
    sub = res[res["Case"] == case].sort_values("AUC", ascending=False)
    for _, r in sub.iterrows():
        mark = " ★" if r["AUC"] == sub["AUC"].max() else ""
        print(f"  {r['Model']:<28} Case{r['Case']:<6} "
              f"{r['AUC']:>8.4f} {r['F1']:>7.4f} "
              f"{r['Recall']:>8.4f} {r['Precision']:>8.4f}{mark}")
    print(f"  {'-'*68}")

best_a = res[res["Case"] == "A"].sort_values("AUC", ascending=False).iloc[0]
best_b = res[res["Case"] == "B"].sort_values("AUC", ascending=False).iloc[0]
print(f"\n  CaseA 최고: {best_a['Model']:<28} AUC={best_a['AUC']:.4f}  Recall={best_a['Recall']:.4f}")
print(f"  CaseB 최고: {best_b['Model']:<28} AUC={best_b['AUC']:.4f}  Recall={best_b['Recall']:.4f}")
print(f"  AUC 차이 (A-B): {best_a['AUC'] - best_b['AUC']:+.4f}")

res.to_excel(OUTPUT_PATH, index=False)
print(f"\n저장 완료: {OUTPUT_PATH}")


  최종 결과 요약 (AUC 내림차순)
  모델                           Case         AUC      F1   Recall     Prec
  GradientBoosting             CaseA        0.6597  0.8026   0.9420   0.6992 ★
  XGBoost(Optuna)              CaseA        0.6577  0.7114   0.6766   0.7500
  LightGBM(Optuna)             CaseA        0.6572  0.7100   0.6737   0.7506
  GradientBoosting(Optuna)     CaseA        0.6559  0.8080   0.9811   0.6869
  LightGBM                     CaseA        0.6457  0.7023   0.6619   0.7480
  LogisticRegression           CaseA        0.6318  0.6735   0.6186   0.7391
  XGBoost                      CaseA        0.6289  0.6963   0.6602   0.7366
  RandomForest                 CaseA        0.6099  0.7651   0.8475   0.6973
  DecisionTree                 CaseA        0.5579  0.6924   0.6821   0.7030
  --------------------------------------------------------------------
  GradientBoosting             CaseB        0.6512  0.8043   0.9530   0.6958 ★
  GradientBoosting(Optuna)     CaseB        0.6506  0.8059